# AI Career Coach: evaluation and comparison charts

This notebook tests the data and the methods used in the project and saves every chart as a PNG you can place in the report.

| Part | Question it answers | Charts |
|---|---|---|
| 1 | What does the training data look like? | profiles per category |
| 2 | Which classifier predicts a career best? | scores by classifier |
| 3 | Which parts of a student profile matter? | scores by feature set |
| 4 | What happens as profiles get sparser? | accuracy against damage |
| 5 | How much training data is needed? | learning curve |
| 6 | Where does the best model go wrong? | accuracy by category, hardest careers |
| 7 | Which search method finds the right career passage? | scores by retriever |
| 8 | Does remembering the conversation help follow-up questions? | with and without memory |

**How to read the numbers.** The student profiles are synthetic: `ml/src/build_training_data.py` generated them from rules. To make the test resemble what real students type, the test profiles are damaged on purpose before the classifiers see them (missing skills, skills from other careers, blank fields). Every classifier score in this notebook is a score on those damaged synthetic profiles and should be quoted that way. Only consented real profiles can support real-world accuracy claims, and the project has none yet.

Every number here comes from running the code. Nothing is typed in by hand.

## Setup

Nothing needs to go into Google Drive. Run the next cell and, when the **Choose Files** button appears, pick `career_coach_evaluation_data.zip` from your computer (it is in `Downloads\AICareerCoach\ml\metrics`). The zip holds the datasets the notebook needs.

To include your fine-tuned retriever in the search comparison (Part 7), set `UPLOAD_FINE_TUNED_MODEL = True` before running. You will then be asked for `career_retriever_model.zip` from the same folder as well. It is about 85 MB, so the upload takes a few minutes; leave it `False` to skip it.

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path

UPLOAD_FINE_TUNED_MODEL = False     # True asks for career_retriever_model.zip as well
DATA_ZIP = "career_coach_evaluation_data.zip"
MODEL_ZIP = "career_retriever_model.zip"
IN_COLAB = "google.colab" in sys.modules


def has_data(folder: Path) -> bool:
    return (folder / "ml" / "data" / "careers" / "career_training_data.csv").is_file()


def unpack(zip_name: str, target: Path) -> bool:
    """Ask for a zip from the computer (unless it is already here) and unpack it into the project folder."""
    if not Path(zip_name).is_file():
        from google.colab import files
        print(f"Choose {zip_name} from your computer.")
        uploaded = files.upload()
        if not uploaded:
            return False
        zip_name = next(iter(uploaded))
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as bundle:
        bundle.extractall(target)
    return True


# When the notebook is run on your own computer, the project folder is found directly.
candidates = [Path(os.environ["AICAREERCOACH_DIR"])] if os.environ.get("AICAREERCOACH_DIR") else []
candidates += [Path("/content/AICareerCoach"), Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_DIR = next((folder for folder in candidates if has_data(folder)), None)

if PROJECT_DIR is None and IN_COLAB:
    PROJECT_DIR = Path("/content/AICareerCoach")
    unpack(DATA_ZIP, PROJECT_DIR)
assert PROJECT_DIR is not None and has_data(PROJECT_DIR), (
    f"The datasets were not found. Run this cell again and choose {DATA_ZIP}.")
PROJECT_DIR = PROJECT_DIR.resolve()

if IN_COLAB and UPLOAD_FINE_TUNED_MODEL and not (PROJECT_DIR / "ml" / "models" / "career-retriever" / "modules.json").is_file():
    unpack(MODEL_ZIP, PROJECT_DIR)

print("Project folder:", PROJECT_DIR)
print("Datasets:", ", ".join(sorted(path.name for path in (PROJECT_DIR / "ml" / "data" / "careers").glob("*.csv"))))
print("Fine-tuned retriever included:", (PROJECT_DIR / "ml" / "models" / "career-retriever" / "modules.json").is_file())

In [ ]:
# Settings you can change.
RANDOM_SEED = 42
NOISE_LEVELS = [0.5, 0.6, 0.7, 0.8]                  # the damage levels compared in Part 4
REALISTIC_NOISE = 0.5                                # how much of each test profile is damaged in the main test
RUN_EMBEDDING_RETRIEVERS = True                      # Part 7: needs sentence-transformers (a few minutes on CPU)
FIGURE_DPI = 200                                     # sharp enough for a printed report

import json, re, time, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
CAREERS_DIR = PROJECT_DIR / "ml" / "data" / "careers"
OUTPUT_DIR = PROJECT_DIR / "ml" / "metrics" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# One look for every chart. The colours are a fixed, colour-blind-safe order: a model or
# method keeps its colour in every chart it appears in.
INK, MUTED, GRID, AXIS = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": INK, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 6,
})
SAVED = []


def save_figure(fig, name: str):
    """Save a chart as a PNG in ml/metrics/evaluation and show it."""
    path = OUTPUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    SAVED.append(path)
    plt.show()
    plt.close(fig)
    print("Saved", path.relative_to(PROJECT_DIR))


def save_table(table: pd.DataFrame, name: str):
    """Save the numbers behind a chart as a CSV, so each chart has a table to quote from."""
    path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    SAVED.append(path)
    return table


def percent_axis(axis, vertical: bool = True):
    (axis.set_ylim if vertical else axis.set_xlim)(0, 100)
    (axis.yaxis if vertical else axis.xaxis).set_major_formatter(matplotlib.ticker.PercentFormatter(decimals=0))

## Part 1. The training data

One row is one student profile, labelled with the career it was generated for. The table's own `split` column fixes which rows are used for training, validation and testing, so results are the same on every run.

In [ ]:
profiles = pd.read_csv(CAREERS_DIR / "career_training_data.csv", encoding="utf-8-sig").fillna("")
catalog = pd.read_csv(CAREERS_DIR / "career_catalog.csv", encoding="utf-8-sig").fillna("")
category_of = dict(zip(catalog["career_id"], catalog["category"]))
title_of = dict(zip(catalog["career_id"], catalog["role_title"]))
profiles["category"] = profiles["target_career_id"].map(category_of)

train = profiles[profiles["split"] == "train"].reset_index(drop=True)
validation = profiles[profiles["split"] == "validation"].reset_index(drop=True)
test = profiles[profiles["split"] == "test"].reset_index(drop=True)

overview = pd.DataFrame({
    "measure": ["profiles", "careers", "categories", "training rows", "validation rows", "test rows",
                "synthetic rows", "real consented rows"],
    "value": [len(profiles), profiles["target_career_id"].nunique(), profiles["category"].nunique(),
              len(train), len(validation), len(test),
              int((profiles["data_origin"] == "synthetic").sum()), int((profiles["data_origin"] != "synthetic").sum())],
})
print(save_table(overview, "01_data_overview").to_string(index=False))

per_category = profiles.groupby("category").agg(profiles=("profile_id", "count"),
                                                careers=("target_career_id", "nunique")).sort_values("profiles")
fig, axis = plt.subplots(figsize=(8, 5.2))
bars = axis.barh(per_category.index, per_category["profiles"], color=SERIES[0], height=0.62)
for bar, careers in zip(bars, per_category["careers"]):
    axis.text(bar.get_width() + 12, bar.get_y() + bar.get_height() / 2,
              f"{int(bar.get_width()):,} profiles, {careers} careers", va="center", color=INK, fontsize=9)
axis.set_xlim(0, per_category["profiles"].max() * 1.42)
axis.grid(axis="y", visible=False)
axis.set_xlabel("Synthetic student profiles")
axis.set_title("Training profiles in each career category")
save_figure(fig, "01_profiles_per_category")
save_table(per_category.reset_index(), "01_profiles_per_category");

## Part 2. Which classifier predicts a career best?

Six classifiers are trained on the same text features (TF-IDF of the profile). The task is hard on purpose: there are 452 possible careers, so guessing would be right about 0.2% of the time.

**The test profiles are made messy on purpose.** Real students leave fields empty, list only some of their skills and mention skills from other careers. A test on tidy generated profiles would say little about that, so the held-out test profiles are damaged in three ways before any classifier sees them. By default half of each profile is damaged (`REALISTIC_NOISE = 0.5`):

- each skill and interest has a 50% chance of being removed (at least one is always kept);
- skills that belong to other careers are added, about half as many as the profile has (seven in ten from a career in the same category, the rest from any field);
- the course, specialisation, certifications and preferred industry are each blanked with a 50% chance.

The classifiers are trained on the training profiles as generated. Every score below is therefore a score on **synthetic profiles with half of their information damaged**, and should be described that way.

Four measures are reported:

- **Accuracy**: the predicted career is the labelled career.
- **Macro precision, recall and F1**: each is worked out for every career and then averaged, so small careers count as much as large ones.

In [ ]:
FEATURE_SETS = {
    "full profile": ["course", "specialisation", "skills", "interests", "experience_level",
                     "certifications", "preferred_industry"],
    "skills and interests": ["skills", "interests"],
    "skills only": ["skills"],
    "interests only": ["interests"],
    "course and specialisation": ["course", "specialisation"],
}
LIST_COLUMNS = ["skills", "interests"]   # columns holding several items separated by semicolons
skills_table = pd.read_csv(CAREERS_DIR / "career_skills.csv", encoding="utf-8-sig").fillna("")
skills_by_career = skills_table.groupby("career_id")["skill"].apply(list).to_dict()
careers_in_category = catalog.groupby("category")["career_id"].apply(list).to_dict()
all_careers = catalog["career_id"].tolist()


def profile_text(frame: pd.DataFrame, columns) -> pd.Series:
    """Join the chosen columns into one text per profile. Each value is tagged with its column."""
    parts = []
    for column in columns:
        tag = column.split("_")[0]
        parts.append(frame[column].astype(str).map(
            lambda value, tag=tag: " ".join(f"{tag}_{re.sub(r'[^a-z0-9]+', '_', item.strip().lower())} {item}"
                                            for item in value.split(";") if item.strip())))
    return pd.concat(parts, axis=1).agg(" ".join, axis=1)


def add_noise(frame: pd.DataFrame, level: float, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Return a damaged copy of the profiles. `level` is between 0 (untouched) and 1."""
    if level <= 0:
        return frame.copy()
    random = np.random.default_rng(seed)
    noisy = frame.copy()
    for column in LIST_COLUMNS:
        values = []
        for text in noisy[column]:
            items = [item.strip() for item in text.split(";") if item.strip()]
            kept = [item for item in items if random.random() >= level] or items[:1]
            values.append("; ".join(kept))
        noisy[column] = values
    distracted = []
    for skills, career_id in zip(noisy["skills"], noisy["target_career_id"]):
        items = [item for item in skills.split("; ") if item]
        neighbours = [other for other in careers_in_category[category_of[career_id]] if other != career_id]
        for _ in range(int(round(level * max(len(items), 3)))):
            # Most borrowed skills come from a neighbouring career; some come from an unrelated field.
            pool = neighbours if random.random() < 0.7 else all_careers
            borrowed = skills_by_career.get(pool[random.integers(len(pool))], [])
            if borrowed:
                items.append(borrowed[random.integers(len(borrowed))])
        distracted.append("; ".join(items))
    noisy["skills"] = distracted
    for column, chance in (("course", level), ("specialisation", level), ("certifications", level),
                           ("preferred_industry", level)):
        noisy.loc[random.random(len(noisy)) < chance, column] = ""
    return noisy


messy_test = add_noise(test, REALISTIC_NOISE)
MESSY = f"messy profiles ({REALISTIC_NOISE:.0%} damaged)"
for column in ("course", "skills", "interests"):
    print(f"As generated {column:9}: {test.loc[0, column]}")
    print(f"As tested    {column:9}: {messy_test.loc[0, column] or '(left empty)'}")


def make_models():
    return {
        "Naive Bayes": MultinomialNB(alpha=0.1),
        "Logistic Regression": LogisticRegression(C=10, max_iter=300),
        "Linear SVM": LinearSVC(C=0.5),
        "k-Nearest Neighbours": KNeighborsClassifier(n_neighbors=7, metric="cosine", weights="distance"),
        "Random Forest": RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=RANDOM_SEED),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_SEED),
    }


MODEL_COLOUR = {name: SERIES[index] for index, name in enumerate(make_models())}
MEASURES = [("accuracy", "Accuracy"), ("precision", "Macro precision"), ("recall", "Macro recall"), ("f1", "Macro F1")]


def evaluate(model, features, labels) -> dict:
    """Accuracy and macro precision, recall and F1, all as percentages."""
    predictions = model.predict(features)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    return {"accuracy": float((predictions == np.asarray(labels)).mean() * 100),
            "precision": float(precision * 100), "recall": float(recall * 100), "f1": float(f1 * 100)}

In [ ]:
print("Training six classifiers on the full profile (the two slowest take about a minute each)...")
full_columns = FEATURE_SETS["full profile"]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
train_features = vectorizer.fit_transform(profile_text(train, full_columns))
models = {}
for name, model in make_models().items():
    started = time.time()
    models[name] = model.fit(train_features, train["target_career_id"])
    print(f"  trained {name} in {time.time() - started:.1f}s")

messy_features = vectorizer.transform(profile_text(messy_test, full_columns))
realistic = pd.DataFrame([{"model": name, "test": MESSY, **evaluate(model, messy_features, test["target_career_id"])}
                          for name, model in models.items()]).round(2)
save_table(realistic, "02_model_comparison")
realistic = realistic.set_index("model")
best_model_name = realistic["accuracy"].idxmax()
print()
print(realistic.drop(columns="test").to_string())
print(f"\nBest on {MESSY}: {best_model_name} ({realistic.loc[best_model_name, 'accuracy']:.1f}% accuracy)")

In [ ]:
order = realistic.sort_values("accuracy", ascending=False).index
fig, axis = plt.subplots(figsize=(10, 4.8))
positions = np.arange(len(order))
for index, (column, label) in enumerate(MEASURES):
    bars = axis.bar(positions + (index - 1.5) * 0.2, realistic.loc[order, column], 0.18, label=label, color=SERIES[index])
    for bar in bars:
        axis.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{bar.get_height():.0f}",
                  ha="center", fontsize=8, color=INK)
axis.set_xticks(positions, [name.replace(" ", "\n", 1) for name in order])
axis.tick_params(axis="x", colors=INK)
axis.grid(axis="x", visible=False)
percent_axis(axis)
axis.set_ylabel(f"Score on {MESSY}")
axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.17), fontsize=9)
axis.set_title(f"Career prediction over 452 careers: synthetic test profiles, {REALISTIC_NOISE:.0%} of each damaged")
save_figure(fig, "02_model_comparison")

## Part 3. Which parts of a profile matter?

The same classifier (Linear SVM, the quickest of the strong ones) is trained on different parts of the profile and scored on the messy test profiles. This shows how much each part contributes and what is lost when a student gives little information.

In [ ]:
ablation_rows = []
for set_name, columns in FEATURE_SETS.items():
    set_vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    set_model = LinearSVC(C=0.5).fit(set_vectorizer.fit_transform(profile_text(train, columns)), train["target_career_id"])
    scores = evaluate(set_model, set_vectorizer.transform(profile_text(messy_test, columns)), test["target_career_id"])
    ablation_rows.append({"features": set_name, **scores})
    print(f"  {set_name}: accuracy {scores['accuracy']:.1f}%, macro F1 {scores['f1']:.1f}%")
ablation = save_table(pd.DataFrame(ablation_rows).round(2), "03_feature_ablation")

fig, axis = plt.subplots(figsize=(8, 3.9))
positions = np.arange(len(ablation))
for index, (column, label) in enumerate([("accuracy", "Accuracy"), ("f1", "Macro F1")]):
    bars = axis.barh(positions + index * 0.36, ablation[column], 0.33, label=label, color=SERIES[index])
    for bar in bars:
        axis.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.1f}%",
                  va="center", fontsize=8.5, color=INK)
axis.set_yticks(positions + 0.18, ablation["features"])
axis.tick_params(axis="y", colors=INK)
axis.invert_yaxis()
axis.grid(axis="y", visible=False)
percent_axis(axis, vertical=False)
axis.set_xlabel(f"Score on {MESSY} (Linear SVM)")
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
axis.set_title("How much each part of the profile contributes")
save_figure(fig, "03_feature_ablation")

## Part 4. How accuracy changes as profiles get sparser

Part 2 damaged half of each profile. This part repeats the test with more of each profile damaged, up to 80%, to show how quickly each classifier loses accuracy as a student gives less usable information, and whether the ranking of the classifiers holds.

This is a stress test on synthetic data. It shows which methods degrade gracefully, not how accurate they would be for real students.

In [ ]:
noise_rows = []
for level in NOISE_LEVELS:
    noisy_features = vectorizer.transform(profile_text(add_noise(test, level), full_columns))
    for name, model in models.items():
        noise_rows.append({"damaged": level, "model": name, **evaluate(model, noisy_features, test["target_career_id"])})
noise_results = save_table(pd.DataFrame(noise_rows).round(2), "04_noise_robustness")
print(noise_results.pivot(index="damaged", columns="model", values="accuracy").round(1).to_string())

fig, axis = plt.subplots(figsize=(8, 4.6))
for name in models:
    series = noise_results[noise_results["model"] == name]
    axis.plot(series["damaged"] * 100, series["accuracy"], marker="o", color=MODEL_COLOUR[name], label=name)
percent_axis(axis)
axis.set_xticks([level * 100 for level in NOISE_LEVELS])
axis.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(decimals=0))
axis.set_xlabel("Share of each test profile that is damaged")
axis.set_ylabel("Accuracy on test profiles")
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, fontsize=9)
axis.set_title("Career prediction as student profiles get sparser")
save_figure(fig, "04_noise_robustness")

## Part 5. How much training data is needed?

Two classifiers are retrained on growing shares of the training data, from about one profile per career up to all of them, and scored on the messy test profiles each time.

In [ ]:
fractions = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
curve_models = ["Linear SVM", "Naive Bayes"]   # the two that train in seconds, so seven sizes stay quick
curve_rows = []
for fraction in fractions:
    subset = pd.concat([group.sample(max(1, int(round(len(group) * fraction))), random_state=RANDOM_SEED)
                        for _, group in train.groupby("target_career_id")])
    subset_vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    subset_features = subset_vectorizer.fit_transform(profile_text(subset, full_columns))
    for name in curve_models:
        model = make_models()[name].fit(subset_features, subset["target_career_id"])
        scores = evaluate(model, subset_vectorizer.transform(profile_text(messy_test, full_columns)), test["target_career_id"])
        curve_rows.append({"model": name, "training_rows": len(subset), "rows_per_career": round(len(subset) / 452, 1),
                           "accuracy": scores["accuracy"], "f1": scores["f1"]})
learning_curve = save_table(pd.DataFrame(curve_rows).round(2), "05_learning_curve")
print(learning_curve.to_string(index=False))

fig, axis = plt.subplots(figsize=(8, 4.4))
for name in curve_models:
    series = learning_curve[learning_curve["model"] == name]
    axis.plot(series["training_rows"], series["accuracy"], marker="o", color=MODEL_COLOUR[name], label=name)
    axis.annotate(f"{series['accuracy'].iloc[-1]:.1f}%", (series["training_rows"].iloc[-1], series["accuracy"].iloc[-1]),
                  textcoords="offset points", xytext=(6, -3), fontsize=9, color=INK)
percent_axis(axis)
axis.set_xlim(0, len(train) * 1.1)
axis.set_xlabel("Training profiles used")
axis.set_ylabel(f"Accuracy on {MESSY}")
axis.legend(loc="lower right", fontsize=9)
axis.set_title("Accuracy as the training set grows")
save_figure(fig, "05_learning_curve")

## Part 6. Where does the best model go wrong?

Using the messy test profiles, this part looks at the errors of the best classifier:

- **accuracy by category** shows which fields are easier and which are harder;
- the **hardest careers** are the ones with the lowest F1 score;
- the **most confused pairs** are the careers most often mistaken for each other.

In [ ]:
best_model = models[best_model_name]
predicted = best_model.predict(messy_features)
actual = test["target_career_id"].to_numpy()
career_accuracy = float((predicted == actual).mean() * 100)

labels = best_model.classes_
per_career_f1 = f1_score(actual, predicted, labels=labels, average=None, zero_division=0) * 100
hardest = pd.DataFrame({"career": [title_of[c] for c in labels], "category": [category_of[c] for c in labels],
                        "f1": per_career_f1}).sort_values("f1").head(15)
save_table(hardest.round(1), "06_hardest_careers")

pairs = Counter((a, p) for a, p in zip(actual, predicted) if a != p)
confused = save_table(pd.DataFrame(
    [{"actual": title_of[a], "predicted": title_of[p], "times": count,
      "same_category": category_of[a] == category_of[p]} for (a, p), count in pairs.most_common(15)]),
    "06_most_confused_pairs")
print(f"{sum(pairs.values())} of {len(actual)} messy test profiles were given the wrong career.")
print(confused.to_string(index=False))

by_category = pd.DataFrame({"category": [category_of[c] for c in actual], "correct": predicted == actual})
by_category = (by_category.groupby("category")["correct"].agg(accuracy="mean", profiles="count")
               .assign(accuracy=lambda frame: frame["accuracy"] * 100).sort_values("accuracy"))
save_table(by_category.reset_index().round(1), "06_accuracy_by_category")
fig, axis = plt.subplots(figsize=(8, 5.2))
bars = axis.barh(by_category.index, by_category["accuracy"], color=SERIES[0], height=0.62)
for bar, count in zip(bars, by_category["profiles"]):
    axis.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.1f}%  ({count} profiles)",
              va="center", fontsize=9, color=INK)
axis.axvline(career_accuracy, color=MUTED, linewidth=1, linestyle="--")
axis.text(career_accuracy, len(by_category) - 0.35, f"all categories {career_accuracy:.1f}%", ha="right", va="bottom",
          fontsize=8.5, color=MUTED)
axis.tick_params(axis="y", colors=INK)
axis.grid(axis="y", visible=False)
percent_axis(axis, vertical=False)
axis.set_xlim(0, 128)
axis.set_xticks(range(0, 101, 20))
axis.set_xlabel(f"Accuracy on {MESSY} ({best_model_name})")
axis.set_title("Right career, by category")
save_figure(fig, "06_accuracy_by_category")

fig, axis = plt.subplots(figsize=(8, 5))
bars = axis.barh(hardest["career"], hardest["f1"], color=SERIES[0], height=0.62)
for bar in bars:
    axis.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.0f}%", va="center", fontsize=9, color=INK)
axis.invert_yaxis()
axis.tick_params(axis="y", colors=INK)
axis.grid(axis="y", visible=False)
percent_axis(axis, vertical=False)
axis.set_xlabel(f"F1 score on {MESSY} ({best_model_name})")
axis.set_title("The 15 careers the model finds hardest to tell apart")
save_figure(fig, "06_hardest_careers")

## Part 7. Which search method finds the right career passage?

The chatbot answers from passages it retrieves from the knowledge base. This part asks questions about every career and checks whether the passages for that career come back.

Five kinds of question:

| Type | Example | Why it is harder |
|---|---|---|
| title | "How do I become a Pharmacist?" | uses the catalogue's own title |
| other name | "What does a Dispensing Pharmacist do?" | uses a different job title |
| skills | "I am good at pharmacology and dispensing. Which career fits?" | no title at all |
| task | "Which career involves work like this: counsel patients on medicines?" | describes the work in plain words |
| with typos | "How do I beocme a Pharamcist?" | a title question typed quickly, with two words misspelt |

Methods compared: **TF-IDF** and **BM25** (keyword search), the **general-purpose embedding model** (all-MiniLM-L6-v2) and the project's **fine-tuned retriever**, if you uploaded it in the setup step.

Measures: **Recall@1** is how often the right career is the first one returned. **MRR** (mean reciprocal rank) also gives part credit when it is second or third.

One caution: the questions are built from the same tables as the passages, so this measures search over the project's own content. It does not measure how well real student questions are answered.

In [ ]:
chunks_path = PROJECT_DIR / "ml" / "data" / "processed" / "chunks.json"
if chunks_path.is_file():
    chunks = json.loads(chunks_path.read_text(encoding="utf-8"))
else:  # build the career passages directly if the pipeline has not been run
    sys.path.insert(0, str(PROJECT_DIR / "ml" / "src"))
    from career_chunks import build_career_chunks
    chunks = build_career_chunks(CAREERS_DIR)
corpus = [chunk["content"] for chunk in chunks]
chunk_resource = np.array([chunk["resource_id"] for chunk in chunks])
print(f"{len(chunks)} passages from {len(set(chunk_resource))} resources "
      f"({sum(r.startswith('career_') for r in set(chunk_resource))} of them careers)")

titles_table = pd.read_csv(CAREERS_DIR / "career_titles.csv", encoding="utf-8-sig").fillna("")
tasks_table = pd.read_csv(CAREERS_DIR / "career_tasks.csv", encoding="utf-8-sig").fillna("")
other_names = titles_table[titles_table["title_type"] == "alternative"].groupby("career_id")["title"].apply(list).to_dict()
tasks_by_career = tasks_table.groupby("career_id")["task"].apply(list).to_dict()
core_skills = skills_table[skills_table["skill_type"] == "core"].groupby("career_id")["skill"].apply(list).to_dict()

random = np.random.default_rng(RANDOM_SEED)
TITLE_QUESTIONS = ["How do I become a {x}?", "What does a {x} do day to day?", "What skills does a {x} need?",
                   "Is {x} a good career and how do I get started?"]
NAME_QUESTIONS = ["What does a {x} do?", "How can I get a job as a {x}?", "What should I study to work as a {x}?"]


def with_typos(text: str, count: int = 2) -> str:
    """Swap two neighbouring letters in `count` of the longer words, as happens when typing quickly."""
    words = text.split()
    candidates = [index for index, word in enumerate(words) if len(word.strip("?.,")) >= 5 and word.strip("?.,").isalpha()]
    for index in random.permutation(candidates)[:count]:
        word = words[index]
        place = int(random.integers(1, len(word.strip("?.,")) - 2))
        words[index] = word[:place] + word[place + 1] + word[place] + word[place + 2:]
    return " ".join(words)


queries = []
for career_id, title in title_of.items():
    gold = f"career_{career_id}"
    if gold not in set(chunk_resource):
        continue
    queries.append(("title", TITLE_QUESTIONS[random.integers(len(TITLE_QUESTIONS))].format(x=title), gold))
    queries.append(("with typos", with_typos(TITLE_QUESTIONS[random.integers(len(TITLE_QUESTIONS))].format(x=title)), gold))
    if other_names.get(career_id):
        name = other_names[career_id][random.integers(len(other_names[career_id]))]
        queries.append(("other name", NAME_QUESTIONS[random.integers(len(NAME_QUESTIONS))].format(x=name), gold))
    if len(core_skills.get(career_id, [])) >= 2:
        picked = random.choice(core_skills[career_id], size=min(3, len(core_skills[career_id])), replace=False)
        queries.append(("skills", f"I am good at {', '.join(picked[:-1])} and {picked[-1]}. Which career fits me?", gold))
    if tasks_by_career.get(career_id):
        task = tasks_by_career[career_id][random.integers(len(tasks_by_career[career_id]))]
        queries.append(("task", f"Which career involves work like this: {task}?", gold))
query_table = pd.DataFrame(queries, columns=["type", "question", "gold"])
QUERY_TYPES = ["title", "other name", "skills", "task", "with typos"]
print(query_table.groupby("type").size().reindex(QUERY_TYPES).to_string())
print(query_table.groupby("type").head(1).to_string(index=False))

In [ ]:
def rank_of_gold(score_matrix, gold_resources, depth: int = 60) -> np.ndarray:
    """Rank of the right resource among the distinct resources returned (1 is best; 0 means not in the first ten)."""
    ranks = np.zeros(len(gold_resources), dtype=int)
    for row, gold in enumerate(gold_resources):
        scores = score_matrix[row]
        scores = np.asarray(scores.todense()).ravel() if sparse.issparse(scores) else np.asarray(scores).ravel()
        seen = []
        for index in np.argsort(-scores)[:depth]:
            resource = chunk_resource[index]
            if resource not in seen:
                seen.append(resource)
                if resource == gold:
                    ranks[row] = len(seen)
                    break
                if len(seen) >= 10:
                    break
    return ranks


def retrieval_scores(ranks: np.ndarray) -> dict:
    """First-result accuracy and mean reciprocal rank, as percentages."""
    return {"recall@1": float((ranks == 1).mean() * 100),
            "mrr": float(np.where(ranks > 0, 1 / np.maximum(ranks, 1), 0).mean() * 100)}


def tfidf_search(questions):
    return (tfidf_search.vectorizer.transform(questions) @ tfidf_search.matrix.T).tocsr()


tfidf_search.vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
tfidf_search.matrix = tfidf_search.vectorizer.fit_transform(corpus)


def bm25_search(questions, k1: float = 1.5, b: float = 0.75):
    return (bm25_search.counter.transform(questions).sign() @ bm25_search.weights.T).tocsr()


bm25_search.counter = CountVectorizer(stop_words="english")
counts = bm25_search.counter.fit_transform(corpus).tocsr().astype(float)
lengths = np.asarray(counts.sum(axis=1)).ravel()
document_frequency = np.asarray((counts > 0).sum(axis=0)).ravel()
inverse_frequency = np.log(1 + (len(corpus) - document_frequency + 0.5) / (document_frequency + 0.5))
coo = counts.tocoo()
normaliser = 1.5 * (1 - 0.75 + 0.75 * lengths[coo.row] / lengths.mean())
bm25_search.weights = sparse.csr_matrix(
    (inverse_frequency[coo.col] * coo.data * 2.5 / (coo.data + normaliser), (coo.row, coo.col)), shape=counts.shape)

RETRIEVERS = {"TF-IDF": tfidf_search, "BM25": bm25_search}

if RUN_EMBEDDING_RETRIEVERS:
    try:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "sentence-transformers"], check=True)
            from sentence_transformers import SentenceTransformer

        def embedding_search(model_source, label):
            model = SentenceTransformer(str(model_source))
            started = time.time()
            passages = model.encode(corpus, batch_size=64, convert_to_numpy=True, normalize_embeddings=True,
                                    show_progress_bar=False)
            print(f"  {label}: embedded {len(corpus)} passages in {time.time() - started:.0f}s")
            return lambda questions: model.encode(list(questions), batch_size=64, convert_to_numpy=True,
                                                  normalize_embeddings=True, show_progress_bar=False) @ passages.T

        RETRIEVERS["General embedding model"] = embedding_search("sentence-transformers/all-MiniLM-L6-v2", "General model")
        for folder, label in (("career-retriever", "Fine-tuned retriever"),
                              ("career-retriever-candidate", "Fine-tuned retriever (candidate)")):
            model_dir = PROJECT_DIR / "ml" / "models" / folder
            if (model_dir / "modules.json").is_file():
                try:
                    RETRIEVERS[label] = embedding_search(model_dir, label)
                except Exception as error:
                    print(f"  {label} could not be loaded and is left out: {str(error)[:160]}")
            else:
                print(f"  {label} was not uploaded, so it is left out.")
    except Exception as error:
        print("Embedding retrievers were skipped:", str(error)[:200])

RETRIEVER_COLOUR = {name: SERIES[index] for index, name in enumerate(
    ["TF-IDF", "BM25", "General embedding model", "Fine-tuned retriever", "Fine-tuned retriever (candidate)"])}
retrieval_rows, RANKS = [], {}
for name, search in RETRIEVERS.items():
    RANKS[name] = rank_of_gold(search(query_table["question"].tolist()), query_table["gold"].tolist())
    retrieval_rows.append({"retriever": name, "questions": len(query_table), **retrieval_scores(RANKS[name])})
overall = save_table(pd.DataFrame(retrieval_rows).round(2), "07_retrieval_comparison").set_index("retriever")
print(overall.to_string())
best_retriever = overall["recall@1"].idxmax()
by_type = {query_type: retrieval_scores(RANKS[best_retriever][(query_table["type"] == query_type).to_numpy()])["recall@1"]
           for query_type in QUERY_TYPES}
hardest_type = min(by_type, key=by_type.get)
print(f"\nHardest kind of question for {best_retriever}: {hardest_type} ({by_type[hardest_type]:.1f}% first-result accuracy)")

In [ ]:
fig, axis = plt.subplots(figsize=(8, 4.4))
positions = np.arange(len(overall))
for index, (column, label) in enumerate([("recall@1", "Recall@1 (right career is the first result)"), ("mrr", "MRR")]):
    bars = axis.bar(positions + (index - 0.5) * 0.36, overall[column], 0.33, label=label, color=SERIES[index])
    for bar in bars:
        axis.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{bar.get_height():.1f}",
                  ha="center", fontsize=9, color=INK)
axis.set_xticks(positions, [name.replace(" ", "\n", 1) for name in overall.index])
axis.tick_params(axis="x", colors=INK)
axis.grid(axis="x", visible=False)
percent_axis(axis)
axis.set_ylabel(f"Score over {len(query_table):,} questions")
axis.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=9)
axis.set_title("Finding the right career passage")
save_figure(fig, "07_retrieval_comparison")

## Part 8. Does remembering the conversation help?

In a conversation, a follow-up such as "continue" or "what skills does it need?" says nothing about the topic by itself. The backend therefore searches with the topic of the conversation when the latest message is a follow-up (`build_retrieval_query` in `backend/app/rag_engine.py`).

This part compares the two tactics. Each test conversation starts with a question about one career and is followed by a follow-up message. The search is run twice, once with the latest message alone, and once with the conversation-aware query. The functions below are a copy of the backend's, so the notebook runs without the backend installed.

In [ ]:
CONTINUE_PHRASES = {
    "continue", "please continue", "continue please", "go on", "go ahead", "carry on", "keep going", "proceed", "more",
    "tell me more", "next", "and then", "then", "finish", "finish it", "resume", "continue from where you left off",
    "continue where you left off", "continue from there", "proceed from where you left off",
    "pick up where you left off", "what next", "what else", "endelea", "endelea tafadhali", "tafadhali endelea",
    "endelea hapo", "halafu", "kisha",
}
REFERENCE_WORDS = {
    "it", "its", "that", "this", "those", "these", "them", "they", "one", "ones", "first", "second", "third", "last",
    "above", "same", "former", "latter", "previous", "earlier", "again", "also", "instead", "more", "another", "other",
    "hiyo", "hilo", "hizo", "hayo", "huyo", "pia", "tena",
}


def is_continuation(message: str) -> bool:
    return re.sub(r"[.!?,…]+$", "", " ".join(message.lower().strip().split())) in CONTINUE_PHRASES


def is_follow_up(message: str) -> bool:
    words = re.findall(r"[a-z']+", message.lower())
    return is_continuation(message) or len(words) <= 6 or any(word in REFERENCE_WORDS for word in words[:12])


def build_retrieval_query(message: str, earlier_student_messages) -> str:
    if not earlier_student_messages or not is_follow_up(message):
        return message
    topic = next((text for text in reversed(earlier_student_messages) if not is_follow_up(text)),
                 earlier_student_messages[-1])[:400]
    return topic if is_continuation(message) else f"{topic} {message}"


FOLLOW_UPS = {
    "\"continue\"": ["continue", "go on", "tell me more", "endelea"],
    "refers back (\"it\", \"that\")": ["What skills does it need?", "How long does that take?", "Is it well paid?",
                                       "Where can I study for it?"],
    "adds an instruction": ["Make it shorter and focus on internships.", "Explain that for a first-year student.",
                            "Give me the same advice but for working abroad."],
}
memory_retriever_name = next((name for name in ("Fine-tuned retriever", "Fine-tuned retriever (candidate)",
                                                "General embedding model", "BM25") if name in RETRIEVERS))
search = RETRIEVERS[memory_retriever_name]
openers = query_table[query_table["type"] == "title"].reset_index(drop=True)
memory_rows = []
for kind, messages in FOLLOW_UPS.items():
    follow_ups = [messages[index % len(messages)] for index in range(len(openers))]
    alone = rank_of_gold(search(follow_ups), openers["gold"].tolist())
    aware = rank_of_gold(search([build_retrieval_query(message, [opener])
                                 for message, opener in zip(follow_ups, openers["question"])]), openers["gold"].tolist())
    memory_rows.append({"follow_up": kind, "tactic": "latest message only", **retrieval_scores(alone)})
    memory_rows.append({"follow_up": kind, "tactic": "conversation-aware", **retrieval_scores(aware)})
memory_results = save_table(pd.DataFrame(memory_rows).round(2), "08_conversation_memory")
print(f"Retriever used: {memory_retriever_name}")
print(memory_results.to_string(index=False))

fig, axis = plt.subplots(figsize=(8, 4.2))
positions = np.arange(len(FOLLOW_UPS))
for index, tactic in enumerate(["latest message only", "conversation-aware"]):
    values = memory_results[memory_results["tactic"] == tactic].set_index("follow_up").reindex(list(FOLLOW_UPS))["recall@1"]
    bars = axis.bar(positions + (index - 0.5) * 0.36, values, 0.33, label=tactic.capitalize(), color=SERIES[index])
    for bar in bars:
        axis.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5, f"{bar.get_height():.0f}%",
                  ha="center", fontsize=9, color=INK)
axis.set_xticks(positions, list(FOLLOW_UPS))
axis.tick_params(axis="x", colors=INK)
axis.grid(axis="x", visible=False)
percent_axis(axis)
axis.set_ylim(0, 114)
axis.set_ylabel("Recall@1: the conversation's career comes first")
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
axis.set_title(f"Follow-up questions with and without conversation memory\n({memory_retriever_name})", fontsize=11)
save_figure(fig, "08_conversation_memory")

## Summary and download

The table below gathers the headline number from each part. All charts and tables are in `ml/metrics/evaluation/` and are also packed into one zip file.

When you write these up, say that the profiles are synthetic and that the test profiles were damaged on purpose, with the damage level used.

In [ ]:
summary = pd.DataFrame([
    {"result": f"Best classifier on {MESSY} (accuracy)", "method": best_model_name, "value": f"{realistic.loc[best_model_name, 'accuracy']:.1f}%"},
    {"result": f"Same classifier on {MESSY} (macro F1)", "method": best_model_name, "value": f"{realistic.loc[best_model_name, 'f1']:.1f}%"},
    {"result": f"Weakest classifier on {MESSY} (accuracy)", "method": realistic["accuracy"].idxmin(), "value": f"{realistic['accuracy'].min():.1f}%"},
    {"result": f"Skills only on {MESSY} (accuracy)", "method": "Linear SVM", "value": f"{ablation.set_index('features').loc['skills only', 'accuracy']:.1f}%"},
    {"result": f"Best classifier with {NOISE_LEVELS[-1]:.0%} of each profile damaged (accuracy)",
     "method": noise_results[np.isclose(noise_results["damaged"], NOISE_LEVELS[-1])].set_index("model")["accuracy"].idxmax(),
     "value": f"{noise_results[np.isclose(noise_results['damaged'], NOISE_LEVELS[-1])]['accuracy'].max():.1f}%"},
    {"result": "Best retriever over all questions (recall@1)", "method": best_retriever, "value": f"{overall.loc[best_retriever, 'recall@1']:.1f}%"},
    {"result": "Hardest kind of question for that retriever (recall@1)", "method": best_retriever, "value": f"{hardest_type}: {by_type[hardest_type]:.1f}%"},
    {"result": "Follow-ups found, latest message only (recall@1)", "method": memory_retriever_name,
     "value": f"{memory_results[memory_results['tactic'] == 'latest message only']['recall@1'].mean():.1f}%"},
    {"result": "Follow-ups found, conversation-aware (recall@1)", "method": memory_retriever_name,
     "value": f"{memory_results[memory_results['tactic'] == 'conversation-aware']['recall@1'].mean():.1f}%"},
])
save_table(summary, "00_summary")
print(summary.to_string(index=False))

(OUTPUT_DIR / "README.txt").write_text(
    "Charts and tables written by ml/notebooks/04_evaluation_and_comparisons_colab.ipynb.\n"
    "All student profiles are synthetic. Scores measure how well each method handles the project's own\n"
    "generated data, with the test profiles deliberately damaged. They are not real-world accuracy.\n", encoding="utf-8")
archive = OUTPUT_DIR.parent / "evaluation_charts.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.iterdir()):
        bundle.write(path, f"evaluation/{path.name}")
print(f"\n{len(list(OUTPUT_DIR.glob('*.png')))} charts and {len(list(OUTPUT_DIR.glob('*.csv')))} tables saved in {OUTPUT_DIR}")
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))